# 05 - Full symmetric grid: RGCN and GAT, all five disciplines (A100)

Builds on the e12 GNN pipeline. Trains RGCN and the cohort-time GAT under the
**HGT tuned protocol** (16-config grid lr x hidden x layers x dropout, selection
on validation AUC-PR, unified budget 200 epochs / patience 15 / weight decay
1e-4, winner at 10 seeds), then the corrected eq. (2) protocol vs M5'.

**Runtime: A100.** **Upload to `MyDrive/`:** `e12_colab_bundle.zip` and
`grid_supplement.zip`. No OpenAlex key is needed (no fetching).

**Ordering (enforced):** one (discipline x architecture) cell at a time,
smallest discipline first (econ, math, physics, neuro, chemistry). Chemistry
RGCN is skipped (already done in notebook 03); its verdict inputs are copied in.
Each finished cell's JSON syncs to `MyDrive/who-inherits/results/robustness/full_symmetric_grid/`
immediately, so partial results are usable at T-1. `DONE_fullgrid.flag` is
written only when all 10 cells and 5 per-discipline verdicts exist.

Safe to re-run after a disconnect: r17 skips completed grid/winner seeds and the
driver skips completed cells; the sync pulls Drive -> local first.


In [ ]:
# Cell 1: deps, mount Drive, unzip both archives to /content/work, determinism
import os, sys, subprocess, zipfile, json, time, threading, shutil
from pathlib import Path
if not os.environ.get("SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch", "torch_geometric==2.6.1", "pandas==2.2.2",
                    "pyarrow", "scikit-learn==1.5.2", "scipy"], check=True)
try:
    from google.colab import drive; drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
except Exception:
    DRIVE = Path(os.environ.get("DRIVE", "."))
    print("not on Colab; DRIVE =", DRIVE)

WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
for z in ("e12_colab_bundle.zip", "grid_supplement.zip"):
    src = DRIVE / z
    assert src.exists(), f"upload {z} to {DRIVE}"
    zipfile.ZipFile(src).extractall(WORK)
os.chdir(WORK)
OUT = WORK / "results" / "robustness" / "full_symmetric_grid"
OUT.mkdir(parents=True, exist_ok=True)
DRIVE_OUT = DRIVE / "who-inherits" / "results" / "robustness" / "full_symmetric_grid"
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
(OUT / "DETERMINISM_NOTE.json").write_text(json.dumps({
    "cudnn_deterministic": True, "use_deterministic_algorithms": "True (warn_only)",
    "note": "PyG hetero scatter ops fall back to nondeterministic CUDA kernels; "
            "per-seed AUC-PR may vary ~1e-3 across reruns. Recorded, not hidden.",
    "torch": torch.__version__}, indent=2))
print("workspace ready at", WORK)


In [ ]:
# Cell 2: dry check, copy chemistry-RGCN inputs, sync helpers (notebook-03 style)
import importlib
FIELDS = ["econ", "math", "physics", "neuro", "chemistry"]   # smallest n_test first
for f in FIELDS:
    os.environ["DATASET"] = f
    sys.path.insert(0, str(WORK / "paper_pipeline"))
    import config as C; importlib.reload(C)
    miss = [n for n, p in (("clean", C.CLEAN_DATASET),
                           ("e1", C.RESULTS_DIR / "e1_baselines.json")) if not os.path.exists(p)]
    print(f"[{f:9s}] {'OK' if not miss else 'MISSING '+','.join(miss)} sha={C.EXPECTED_SHA256[:12]}")
    assert not miss, f"{f}: {miss}"

# chemistry RGCN already done: place its per-seed files where r17 aggregate reads them
for p in (WORK / "chemistry_rgcn_existing").glob("chemistry_rgcn_sym_seed*.json"):
    shutil.copy2(p, OUT / p.name)
shutil.copy2(WORK / "chemistry_rgcn_existing" / "rgcn_symmetric_verdict.json",
             OUT / "chemistry_rgcn_verdict_from_r3.json")
print("copied chemistry RGCN inputs:", len(list(OUT.glob('chemistry_rgcn_sym_seed*.json'))), "seeds")

def sync():
    subprocess.run(["rsync", "-a", str(OUT) + "/", str(DRIVE_OUT) + "/"], check=False)
    # rsync may be absent on Colab; fall back to copy
    if not any(DRIVE_OUT.iterdir()):
        for p in OUT.glob("*"):
            if p.is_file(): shutil.copy2(p, DRIVE_OUT / p.name)
    print(f"[sync] {time.strftime('%H:%M:%S')} -> {DRIVE_OUT}")

_stop = threading.Event()
def _ckpt():
    while not _stop.wait(600):
        try: sync()
        except Exception as e: print("[sync] checkpoint failed:", e)
threading.Thread(target=_ckpt, daemon=True).start()
# pull anything a prior session synced (resume)
for p in DRIVE_OUT.glob("*"):
    if p.is_file() and not (OUT / p.name).exists(): shutil.copy2(p, OUT / p.name)
print("sync helpers ready")


In [ ]:
# Cell 3: driver - one (discipline x architecture) cell at a time, smallest first.
# Chemistry RGCN skipped (copied in). Each finished cell syncs immediately; the
# per-discipline eq.(2) verdict runs once both architectures for that field exist.
def run(cmd_args, field):
    env = {**os.environ, "DATASET": field}
    r = subprocess.run([sys.executable, str(WORK / "r17_full_symmetric_grid.py"),
                        *cmd_args], cwd=str(WORK), env=env)
    if r.returncode: raise RuntimeError(f"{field} {cmd_args} exited {r.returncode}")

def winner_done(field, model):
    return all((OUT / f"{field}_{model}_sym_seed{s}.json").exists() for s in range(10))

for field in FIELDS:
    for model in ("rgcn", "gat"):
        if field == "chemistry" and model == "rgcn":
            print(f"[skip] chemistry rgcn (done in nb03)"); continue
        if winner_done(field, model):
            print(f"[skip] {field} {model} winner already complete"); continue
        print(f"==== TRAIN {field} {model} ====", flush=True)
        run(["train", "--model", model], field)
        sync()                                   # partial result usable at T-1
    if not (OUT / f"{field}_verdict.json").exists():
        print(f"==== AGGREGATE {field} ====", flush=True)
        run(["aggregate"], field)
        v = json.load(open(OUT / f"{field}_verdict.json"))
        print(field, "verdict:", {m: d["exceeds_fair"] for m, d in v["models"].items()})
        sync()
_stop.set(); sync()
print("all cells done")


In [ ]:
# Cell 4: DONE flag only when all 10 cells + 5 verdicts exist
need = [(f, m) for f in FIELDS for m in ("rgcn", "gat")]
cells_ok = all(winner_done(f, m) or (f == "chemistry" and m == "rgcn") for f, m in need)
verdicts_ok = all((OUT / f"{f}_verdict.json").exists() for f in FIELDS)
assert cells_ok and verdicts_ok, f"incomplete: cells={cells_ok} verdicts={verdicts_ok}"
flag = {"job": "fullgrid", "finished_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "verdicts": {f: json.load(open(OUT / f"{f}_verdict.json"))["models"] for f in FIELDS}}
(OUT / "DONE_fullgrid.flag").write_text(json.dumps(flag, indent=2))
sync()
(DRIVE / "who-inherits" / "DONE_fullgrid.flag").write_text(json.dumps(flag, indent=2))
print("DONE_fullgrid.flag written")
